# Reversible Reactions, All, and Any

Three patterns that simplify common tasks: tuple rates with `@` for reversible reactions, `All` for targeting every state, and `Any` for applying characteristics to blocks of reactions.

In [ ]:
from mobspy import *

Reversible reactions are written with `@` and a tuple of (forward, backward) rates.

In [ ]:
A, B = BaseSpecies()

A >> B @ (1, 0.5)

A(100)
B(0)

S = Simulation(A | B)
S.duration = 10
print(S.compile(verbose=True))

The compiled output shows two reactions: one converting A to B at rate 1, and a reverse reaction converting B to A at rate 0.5.

Passing a tuple of two rates to `@` creates a reversible reaction with forward and backward rates.

In [ ]:
C, D = BaseSpecies()

C >> D @ (1, 0.5)

C(100)
D(0)

S2 = Simulation(C | D)
S2.duration = 10
print(S2.compile(verbose=True))

`All[]` targets every state of a species. It is useful for setting initial counts across all states and for creating reactions that produce every state.

In [ ]:
S_state = BaseSpecies()
S_state.alive
S_state.dead

# Set 10 copies in every state
All[S_state](10)

S3 = Simulation(S_state)
S3.duration = 10
print(S3.compile(verbose=True))

In reactions, `All` on the product side means the reaction produces all states of that species.

In [ ]:
E, F = BaseSpecies()
F.red
F.blue

# E produces both F.red and F.blue
E >> All[F] @ 1

E(50)

S4 = Simulation(E | F)
S4.duration = 10
print(S4.compile(verbose=True))

`Any` is a context manager that adds characteristics to all species in the reactions inside its block. This avoids repeating `.red`, `.big`, etc. on every species.

In [ ]:
Color = BaseSpecies()
Color.red
Color.blue

G = BaseSpecies()
H = BaseSpecies()
G_colored = G * Color
H_colored = H * Color

with Any.red:
    G_colored >> H_colored @ 0.5

G_colored.red(100)

S5 = Simulation(G_colored | H_colored)
S5.duration = 10
print(S5.compile(verbose=True))

The reaction inside the `with Any.red:` block applies `.red` to both G and H. Only `G.red` reacts, and it produces `H.red`.

`Any` contexts can be nested. Each level adds its own characteristic query.

In [ ]:
Color2 = BaseSpecies()
Color2.red
Color2.blue

Size = BaseSpecies()
Size.big
Size.small

P_base = BaseSpecies()
Q_base = BaseSpecies()
P = P_base * Color2 * Size
Q = Q_base * Color2 * Size

with Any.red:
    with Any.big:
        P >> Q @ 1

P.red.big(50)

S6 = Simulation(P | Q)
S6.duration = 10
print(S6.compile(verbose=True))